In [26]:
from sls_client import get_sls_data_by_query
from datetime import datetime, timedelta
import pandas as pd

# 设置pandas显示选项以展示更多内容
pd.set_option("display.max_rows", 100)  # 显示最多100行
pd.set_option("display.max_columns", None)  # 显示所有列
pd.set_option("display.width", 1000)  # 设置显示宽度
pd.set_option("display.max_colwidth", 100)  # 设置列最大宽度

import sqlite3


def get_user_variant_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_user_variant.db"
    table_name = f"search_ab_user_variant_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    query = f"""
type:a and ap:/product/ and pageName:/search/goods|
select regexp_replace(ap, '\d+','{{digit}}') as api,pageName as page_ame,
json_extract_scalar(json_extract_scalar(ai, '$.qh.xm-ab-exp'),'$[1].experimentId') experiment_id,
type,uid,date_format(__time__, '%Y%m%d') as ds,count(1) search_times,
array_join(array_sort(array_agg(distinct json_extract_scalar(json_extract_scalar(ai, '$.qh.xm-ab-exp'),'$[1].variantId'))),',') variant_list
from log group by 1,2,3,4,5,6 limit 1000000"""
    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    _df = get_sls_data_by_query(
        query=query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
        from_time=from_time,
        to_time=to_time,
    )

    _df["search_times"] = _df["search_times"].fillna(1).astype(int)

    _df["variant_list"] = _df["variant_list"].fillna("none")

    if not _df.empty:
        _df.drop(columns=["__source__", "__time__"], inplace=True)
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return _df


all_user_variant_df = pd.DataFrame()
start_date = datetime(2025, 1, 1)
end_date = datetime.now()
current_date = start_date
while current_date <= end_date:
    check_if_exists = current_date.strftime("%Y%m%d") != end_date.strftime("%Y%m%d")
    df = get_user_variant_of_date_from_sls(
        current_date, check_if_local_exist=check_if_exists
    )
    all_user_variant_df = pd.concat([all_user_variant_df, df], ignore_index=True)
    current_date += timedelta(days=1)

all_user_variant_df.head(10)

即将获取数据: =====> 2025-01-04 00:00:00 2025-01-04 23:59:59.999999 xm-mall: 
type:a and ap:/product/ and pageName:/search/goods|
select regexp_replace(ap, '\d+','{digit}') as a
>=====数条数:6721
即将获取数据: =====> 2025-01-05 00:00:00 2025-01-05 23:59:59.999999 xm-mall: 
type:a and ap:/product/ and pageName:/search/goods|
select regexp_replace(ap, '\d+','{digit}') as a
>=====数条数:3319


,api,page_ame,experiment_id,type,uid,ds,search_times,variant_list
0,/product/{digit}/{digit},/search/goods,product_search_rerank,a,376160,20250101,1,V4
1,/product/{digit}/{digit},/search/goods,product_search_rerank,a,156155,20250101,2,V2
2,/product/{digit}/{digit},/search/goods,product_search_rerank,a,228312,20250101,3,V3
3,/product/{digit}/{digit},/search/goods,product_search_rerank,a,334838,20250101,2,V5
4,/product/{digit}/{digit},/search/goods,product_search_rerank,a,415677,20250101,5,V1
5,/product/{digit}/{digit},/search/goods,product_search_rerank,a,495427,20250101,3,V5
6,/product/{digit}/{digit},/search/goods,product_search_rerank,a,357255,20250101,2,V2
7,/product/{digit}/{digit},/search/goods,product_search_rerank,a,250633,20250101,20,V3
8,/product/{digit}/{digit},/search/goods,product_search_rerank,a,440577,20250101,4,V4
9,/product/{digit}/{digit},/search/goods,product_search_rerank,a,471591,20250101,6,V3


In [27]:
# idx:4,name:徐州奶油草莓 净重3-3.2斤/一级/单果10g+/板装,pid:goods,sku:5442468008,salePrice:69.5,pdid:702,stock:10000,ext:cross;idx:5,name:徐州奶油草莓 280G*1盒/一级/4*6/ ,pid:goods,sku:5442468073,salePrice:16.5,pdid:702,stock:10000,ext:cross;idx:6,name:徐州奶油草莓 净重2.8-3斤/一级/单果10g+/ 10盒,pid:goods,sku:5442468518,salePrice:74.5,pdid:702,stock:10000,ext:cross

view_query = """
type:view and pageName:/search/goods |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx,
  regexp_extract(sku_item, 'name:([^,]+)', 1) AS name,
  regexp_extract(sku_item, 'sku:([\dA-Z]+)', 1) AS sku,
  regexp_extract(sku_item, 'pid:([^,]+)', 1) AS pid,
  regexp_extract(sku_item, 'pdid:(\d+)', 1) AS pdid,
* from(
select uid,date_format(__time__, '%Y%m%d') ds,url,bid_list.sku_item,
url_extract_parameter(split_part(url,'#/',2), 'pdName') AS search_query,rqCount rq_count,type
from log,unnest(split(bid,';')) as bid_list(sku_item) limit 10000000)
having pid = 'goods'
"""


def get_user_sku_view_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_user_sku_view.db"
    table_name = f"user_sku_view_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    _df = get_sls_data_by_query(
        query=view_query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
        from_time=from_time,
        to_time=to_time,
    )

    if not _df.empty:
        _df.drop(columns=["__source__", "__time__"], inplace=True)
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return _df


all_user_sku_view_df = pd.DataFrame()
start_date = datetime(2025, 1, 1)
end_date = datetime.now()
current_date = start_date
while current_date <= end_date:
    check_if_exists = current_date.strftime("%Y%m%d") != end_date.strftime("%Y%m%d")
    df = get_user_sku_view_of_date_from_sls(
        current_date, check_if_local_exist=check_if_exists
    )
    all_user_sku_view_df = pd.concat([all_user_sku_view_df, df], ignore_index=True)
    current_date += timedelta(days=1)

all_user_sku_view_df.head(10)

即将获取数据: =====> 2025-01-04 00:00:00 2025-01-04 23:59:59.999999 xm-mall: 
type:view and pageName:/search/goods |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx,
  
>=====数条数:224357
即将获取数据: =====> 2025-01-05 00:00:00 2025-01-05 23:59:59.999999 xm-mall: 
type:view and pageName:/search/goods |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx,
  
>=====数条数:95597


,idx,name,sku,pid,pdid,uid,ds,url,sku_item,search_query,rq_count,type
0,0,丹东红颜草莓 250G*1盒/一级/4*6,17120703160,goods,2586,500151,20250101,https://h5.summerfarm.net/home.html?token=mall__6bf7fab8-8688-4f3a-8f7d-f875bbb6a218&time=173566...,"idx:0,name:丹东红颜草莓 250G*1盒/一级/4*6,pid:goods,sku:17120703160,salePrice:23,pdid:2586,stock:10000,ex...",草莓,139,view
1,1,安徽红颜草莓 净重3-3.2斤/一级/单果8g+,555812873784,goods,4760,500151,20250101,https://h5.summerfarm.net/home.html?token=mall__6bf7fab8-8688-4f3a-8f7d-f875bbb6a218&time=173566...,"idx:1,name:安徽红颜草莓 净重3-3.2斤/一级/单果8g+,pid:goods,sku:555812873784,salePrice:89,pdid:4760,stock:0,ex...",草莓,139,view
2,2,丹东红颜草莓 250G*1盒/一级/4*5,17120703004,goods,2586,500151,20250101,https://h5.summerfarm.net/home.html?token=mall__6bf7fab8-8688-4f3a-8f7d-f875bbb6a218&time=173566...,"idx:2,name:丹东红颜草莓 250G*1盒/一级/4*5,pid:goods,sku:17120703004,salePrice:22,pdid:2586,stock:10000,ex...",草莓,139,view
3,3,徐州奶油草莓 净重3-3.2斤/一级/单果10g+/板装,5442468008,goods,702,500151,20250101,https://h5.summerfarm.net/home.html?token=mall__6bf7fab8-8688-4f3a-8f7d-f875bbb6a218&time=173566...,"idx:3,name:徐州奶油草莓 净重3-3.2斤/一级/单果10g+/板装,pid:goods,sku:5442468008,salePrice:78,pdid:702,stock:0,e...",草莓,140,view
4,4,徐州奶油草莓 280G*1盒/一级/4*5/,5442468100,goods,702,500151,20250101,https://h5.summerfarm.net/home.html?token=mall__6bf7fab8-8688-4f3a-8f7d-f875bbb6a218&time=173566...,"idx:4,name:徐州奶油草莓 280G*1盒/一级/4*5/ ,pid:goods,sku:5442468100,salePrice:18,pdid:702,stock:10000,ex...",草莓,141,view
5,17,越南大青芒（5-6成熟） 净重52-54斤/二级/单果500g+(（放熟后产生的损耗不予售后）),533164186415,goods,9905,62681,20250101,https://h5.summerfarm.net/home.html?token=mall__49c1bb64-ac30-43c7-8921-9b7f38c4a124&time=173566...,"idx:17,name:越南大青芒（5-6成熟） 净重52-54斤/二级/单果500g+(（放熟后产生的损耗不予售后）),pid:goods,sku:533164186415,salePric...",null,95,view
6,0,C味糯米小圆子 1KG*1包,664755736811,goods,3828,209604,20250101,https://h5.summerfarm.net/home.html?code=061U8Tkl2ELvNe4kiXll2vDrb90U8Tkz&state=STATE#/search/go...,"idx:0,name:C味糯米小圆子 1KG*1包,pid:goods,sku:664755736811,salePrice:19,pdid:3828,stock:10000,ext:cross",糯米,40,view
7,1,C味糯米小圆子 1KG*10包,664755736063,goods,3828,209604,20250101,https://h5.summerfarm.net/home.html?code=061U8Tkl2ELvNe4kiXll2vDrb90U8Tkz&state=STATE#/search/go...,"idx:1,name:C味糯米小圆子 1KG*10包,pid:goods,sku:664755736063,salePrice:166,pdid:3828,stock:10000,ext:cross",糯米,40,view
8,2,三象水磨糯米粉 500G*20包,3807662084,goods,960,209604,20250101,https://h5.summerfarm.net/home.html?code=061U8Tkl2ELvNe4kiXll2vDrb90U8Tkz&state=STATE#/search/go...,"idx:2,name:三象水磨糯米粉 500G*20包,pid:goods,sku:3807662084,salePrice:157,pdid:960,stock:10000,ext:cross",糯米,40,view
9,3,C味血糯米罐头 900G*12罐,713337205160,goods,4758,209604,20250101,https://h5.summerfarm.net/home.html?code=061U8Tkl2ELvNe4kiXll2vDrb90U8Tkz&state=STATE#/search/go...,"idx:3,name:C味血糯米罐头 900G*12罐,pid:goods,sku:713337205160,salePrice:135,pdid:4758,stock:10000,ext:c...",糯米,40,view


In [28]:
from sls_client import get_sls_raw_data_by_query

click_query = """
type:cl and pageName:/search/goods |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx,
  regexp_extract(sku_item, 'name:([^,]+)', 1) AS name,
  regexp_extract(sku_item, 'sku:([\dA-Z]+)', 1) AS sku,
  regexp_extract(sku_item, 'pid:([^,]+)', 1) AS pid,
  regexp_extract(sku_item, 'pdid:(\d+)', 1) AS pdid,
ds,search_query,type,uid,page_name from(
select uid,date_format(__time__, '%Y%m%d') ds,url,bid_list.sku_item,pageName as page_name,
url_extract_parameter(split_part(url,'#/',2), 'pdName') AS search_query,rqCount rq_count,type
from log,unnest(split(bid,';')) as bid_list(sku_item) limit 10000000)"""

click_query = "type:cl and pageName:/search/goods"


def get_user_sku_click_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_user_sku_click.db"
    table_name = f"user_sku_click_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    offset = 0
    all_df = pd.DataFrame()
    while True:
        _df = get_sls_raw_data_by_query(
            query=click_query,
            project="xianmu-front-end-log",
            logstore="xm-mall",
            from_time=from_time,
            to_time=to_time,
            offset=offset,
            line=100,
        )
        if _df.empty:
            break
        all_df = pd.concat([all_df, _df], ignore_index=True)
        if len(_df) < 100:
            break
        offset += 100
    _df = all_df

    if not _df.empty:
        _df.drop(
            columns=[
                "__source__",
                "__time__",
                "userAgent",
                "url",
                "__topic__",
                "__tag__:__client_ip__",
                "__tag__:__receive_time__",
                "__time_ns_part__",
            ],
            inplace=True,
        )
        _df["ds"] = day.strftime("%Y%m%d")
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return _df


all_user_sku_click_df = pd.DataFrame()
start_date = datetime(2025, 1, 1)
end_date = datetime.now()
current_date = start_date
while current_date <= end_date:
    check_if_exists = current_date.strftime("%Y%m%d") != end_date.strftime("%Y%m%d")
    df = get_user_sku_click_of_date_from_sls(
        current_date, check_if_local_exist=check_if_exists
    )
    all_user_sku_click_df = pd.concat([all_user_sku_click_df, df], ignore_index=True)
    current_date += timedelta(days=1)

all_user_sku_click_df.head(10)

即将获取数据: =====>from_time:2025-01-04 00:00:00, to_time:2025-01-04 23:59:59.999999, logstore:xm-mall, query:type:cl and pageName:/search/goods
>=====数据条数:100
即将获取数据: =====>from_time:2025-01-04 00:00:00, to_time:2025-01-04 23:59:59.999999, logstore:xm-mall, query:type:cl and pageName:/search/goods
>=====数据条数:100
即将获取数据: =====>from_time:2025-01-04 00:00:00, to_time:2025-01-04 23:59:59.999999, logstore:xm-mall, query:type:cl and pageName:/search/goods
>=====数据条数:100
即将获取数据: =====>from_time:2025-01-05 00:00:00, to_time:2025-01-05 23:59:59.999999, logstore:xm-mall, query:type:cl and pageName:/search/goods
>=====数据条数:100


,time,v,title,cid,uid,phone,uName,en,level,type,pageName,sid,rqCount,linkInfo,tag,pid,idx,name,sku,pdid,stock,ext,bid,abTestVersion,ds
0,1735660813055,0.1.32,鲜沐农场,1735660805341-433944,337640,18307905065,幸运蛋糕青山湖店,wbv,l,cl,/search/goods,1735660805340-315466,44,"name:searchGoods,pdName:安佳,type:2",1,唤起购买,2,安佳淡奶油 1L*12盒,N001S01R005,56,10000,唤起加购弹窗,undefined,V4,20250101
1,1735660814606,0.1.32,鲜沐农场,1735660805341-433944,337640,18307905065,幸运蛋糕青山湖店,wbv,l,cl,/search/goods,1735660805340-315466,51,"name:searchGoods,pdName:安佳,type:2",1,加购弹窗,None,步进器,N001S01R005,None,2,None,undefined,V4,20250101
2,1735660816328,0.1.32,鲜沐农场,1735660805341-433944,337640,18307905065,幸运蛋糕青山湖店,wbv,l,cl,/search/goods,1735660805340-315466,53,"name:searchGoods,pdName:安佳,type:2",1,None,None,None,None,None,None,None,"name:加入购物车,pid:加购弹窗,sku:N001S01R005,pdid:56,stock:1980",V4,20250101
3,1735660815383,0.1.32,鲜沐农场,1735660687308-26940,453425,15006213532,汉堡滨海吾悦,web,l,cl,/search/goods,1735660783225-751292,46,"name:searchGoods,pdName:榨汁橙,type:2",1,None,None,None,None,None,None,None,"idx:0,name:榨汁橙 净重33-35斤/普通/单果100g+,pid:goods,sku:6null60520162,salePrice:86,pdid:429,stock:10000...",V3,20250101
4,1735660822487,0.1.32,鲜沐农场,1734100694927-292719,501851,18100222440,拉丁茂广场霸王茶姬,wbv,l,cl,/search/goods,1735660753189-165447,127,"name:searchGoods,pdName:柠檬,type:2",1,唤起购买,0,广东粗皮香水柠檬 10斤*1包/一级/80-130g/-(霸王茶姬专用),16788463274,1572,10000,唤起加购弹窗,undefined,V3,20250101
5,1735660826464,0.1.32,鲜沐农场,1734100694927-292719,501851,18100222440,拉丁茂广场霸王茶姬,wbv,l,cl,/search/goods,1735660753189-165447,134,"name:searchGoods,pdName:柠檬,type:2",1,None,None,None,None,None,None,None,"name:加入购物车,pid:加购弹窗,sku:16788463274,pdid:1572,stock:167",V3,20250101
6,1735660843405,0.1.32,鲜沐农场,1734869360697-945513,498537,15113756178,益禾堂谢岗吓角店,wbv,l,cl,/search/goods,1735660805397-652642,60,"name:searchGoods,pdName:青稞,type:2",1,None,None,None,None,None,None,None,"idx:2,name:乔翊娅糖水青稞罐头 900G*12罐,pid:goods,sku:713137118123,salePrice:106,pdid:10279,stock:10000,ex...",V4,20250101
7,1735660864275,0.1.32,鲜沐农场,1734869360697-945513,498537,15113756178,益禾堂谢岗吓角店,wbv,l,cl,/search/goods,1735660805397-652642,100,"name:searchGoods,pdName:青稞,type:2",1,唤起购买,2,乔翊娅糖水青稞罐头 900G*12罐,713137118123,10279,10000,唤起加购弹窗,undefined,V4,20250101
8,1735660865277,0.1.32,鲜沐农场,1735660501175-363302,460667,19185660527,熊猫别汤,wbv,l,cl,/search/goods,1735660837092-562376,65,"name:searchGoods,pdName:爱乐薇(铁塔)淡奶油,type:2",1,None,None,None,None,None,None,None,"idx:0,name:爱乐薇(铁塔)淡奶油 1L*12盒,pid:goods,sku:N001S01R002,salePrice:590,pdid:52,stock:10000,ext:cross",V4,20250101
9,1735660866412,0.1.32,鲜沐农场,1734869360697-945513,498537,15113756178,益禾堂谢岗吓角店,wbv,l,cl,/search/goods,1735660805397-652642,108,"name:searchGoods,pdName:青稞,type:2",1,加购弹窗,None,步进器,713137118123,None,2,None,undefined,V4,20250101


In [29]:
import re

all_user_sku_click_explored = []
pattern = re.compile(r'idx:(?P<idx>\d+).*?name:(?P<name>[^,]+).*?pid:(?P<pid>[^,]+).*?sku:(?P<sku>[^,]+).*?pdid:(?P<pdid>[^,]+)')

for index, row in all_user_sku_click_df.iterrows():
    _dict = row.to_dict()
    bid = _dict["bid"]
    for bid_item in bid.split(";"):
        sku_info = {}
        sku_info.update(_dict)
        search_query=_dict["linkInfo"]
        sku_info["search_query"] = re.search(r'pdName:([^,]+)', search_query).group(1)
        sku_info["bid"] = bid_item
        if 'undefined' in bid_item:
            all_user_sku_click_explored.append(sku_info)
        else:
            try:
                idx = pdid = sku = pid = name = None
                
                idx_match = re.search(r'idx:(\d+)', bid_item)
                if idx_match:
                    idx = idx_match.group(1)
                    
                pdid_match = re.search(r'pdid:(\d+)', bid_item)
                if pdid_match:
                    pdid = pdid_match.group(1)
                    
                sku_match = re.search(r'sku:([\dA-Za-z]+)', bid_item)
                if sku_match:
                    sku = sku_match.group(1)
                    
                pid_match = re.search(r'pid:([^,]+)', bid_item)
                if pid_match:
                    pid = pid_match.group(1)
                    
                name_match = re.search(r'name:([^,]+)', bid_item)
                if name_match:
                    name = name_match.group(1)
                    
                sku_info.update({
                    "idx": idx,
                    "pdid": pdid, 
                    "sku": sku,
                    "pid": pid,
                    "name": name
                })
                all_user_sku_click_explored.append(sku_info)
            except Exception as e:
                print(e, bid_item)
                raise e

all_user_sku_click_explored_df = pd.DataFrame(all_user_sku_click_explored)
all_user_sku_click_explored_df[['bid','sku','name','idx','pid','pdid','linkInfo','search_query']].head(5)

,bid,sku,name,idx,pid,pdid,linkInfo,search_query
0,undefined,N001S01R005,安佳淡奶油 1L*12盒,2,唤起购买,56,"name:searchGoods,pdName:安佳,type:2",安佳
1,undefined,N001S01R005,步进器,None,加购弹窗,None,"name:searchGoods,pdName:安佳,type:2",安佳
2,"name:加入购物车,pid:加购弹窗,sku:N001S01R005,pdid:56,stock:1980",N001S01R005,加入购物车,None,加购弹窗,56,"name:searchGoods,pdName:安佳,type:2",安佳
3,"idx:0,name:榨汁橙 净重33-35斤/普通/单果100g+,pid:goods,sku:6null60520162,salePrice:86,pdid:429,stock:10000...",6null60520162,榨汁橙 净重33-35斤/普通/单果100g+,0,goods,429,"name:searchGoods,pdName:榨汁橙,type:2",榨汁橙
4,undefined,16788463274,广东粗皮香水柠檬 10斤*1包/一级/80-130g/-(霸王茶姬专用),0,唤起购买,1572,"name:searchGoods,pdName:柠檬,type:2",柠檬
5,"name:加入购物车,pid:加购弹窗,sku:16788463274,pdid:1572,stock:167",16788463274,加入购物车,None,加购弹窗,1572,"name:searchGoods,pdName:柠檬,type:2",柠檬
6,"idx:2,name:乔翊娅糖水青稞罐头 900G*12罐,pid:goods,sku:713137118123,salePrice:106,pdid:10279,stock:10000,ex...",713137118123,乔翊娅糖水青稞罐头 900G*12罐,2,goods,10279,"name:searchGoods,pdName:青稞,type:2",青稞
7,undefined,713137118123,乔翊娅糖水青稞罐头 900G*12罐,2,唤起购买,10279,"name:searchGoods,pdName:青稞,type:2",青稞
8,"idx:0,name:爱乐薇(铁塔)淡奶油 1L*12盒,pid:goods,sku:N001S01R002,salePrice:590,pdid:52,stock:10000,ext:cross",N001S01R002,爱乐薇(铁塔)淡奶油 1L*12盒,0,goods,52,"name:searchGoods,pdName:爱乐薇(铁塔)淡奶油,type:2",爱乐薇(铁塔)淡奶油
9,undefined,713137118123,步进器,None,加购弹窗,None,"name:searchGoods,pdName:青稞,type:2",青稞


In [30]:
print(all_user_variant_df.columns)
print(all_user_sku_view_df.columns)
print(all_user_sku_click_df.columns)

all_sku_view_data_df = all_user_sku_view_df[
    [
        "idx",
        "name",
        "sku",
        "sku_item",
        "pid",
        "pdid",
        "uid",
        "ds",
        "search_query",
        "type",
    ]
].merge(
    all_user_variant_df[["uid", "ds", "variant_list", "search_times"]],
    on=["uid", "ds"],
    how="left",
)


Index(['api', 'page_ame', 'experiment_id', 'type', 'uid', 'ds', 'search_times', 'variant_list'], dtype='object')
Index(['idx', 'name', 'sku', 'pid', 'pdid', 'uid', 'ds', 'url', 'sku_item', 'search_query', 'rq_count', 'type'], dtype='object')
Index(['time', 'v', 'title', 'cid', 'uid', 'phone', 'uName', 'en', 'level', 'type', 'pageName', 'sid', 'rqCount', 'linkInfo', 'tag', 'pid', 'idx', 'name', 'sku', 'pdid', 'stock', 'ext', 'bid', 'abTestVersion', 'ds'], dtype='object')


In [31]:
all_user_sku_click_explored_df.groupby("pid").size().reset_index(name="count").sort_values(
    "count", ascending=False
).head(10)

,pid,count
1,加购弹窗,41027
3,唤起购买,35191
0,goods,30684
2,唤起提醒,43


In [32]:
user_click_with_variant_df = all_user_sku_click_explored_df.merge(
    all_user_variant_df[["uid", "ds", "variant_list"]],
    on=["uid", "ds"],
    how="left",
)

user_click_with_variant_df["action_type"] = user_click_with_variant_df.apply(
    lambda row: (
        "加入购物车"
        if row["pid"] == "加购弹窗" and row["name"] == "加入购物车"
        else "商品详情" if row["pid"] == "goods" else row["pid"]
    ),
    axis=1,
)
user_click_with_variant_df['idx']=user_click_with_variant_df['idx'].fillna(-1).astype(int)

In [33]:
import pandasql

user_click_with_variant_statistics_df = pandasql.sqldf("""
select uid,variant_list,ds,count(case when action_type='商品详情' then 1 end) as 商品详情cnt,
count(case when action_type='加入购物车' then 1 end) as 加入购物车cnt,
count(case when action_type='唤起购买' and variant_list is not null then 1 end) as 唤起购买cnt,
round(avg(case when action_type='商品详情' or action_type='唤起购买' then idx end),1) as avg点击位置,
max(case when action_type='商品详情' or action_type='唤起购买' then idx end) as max点击位置,
min(case when action_type='商品详情' or action_type='唤起购买' then idx end) as min点击位置,
count(distinct sku) 点击SKU_cnt,
count(distinct search_query) 搜索词cnt                        
from user_click_with_variant_df group by uid,variant_list,ds
""")


In [34]:
all_sku_view_data_df["idx"] = all_sku_view_data_df["idx"].fillna(-1).astype(int)
user_view_with_variant_statistics_df = pandasql.sqldf(
"""
select uid,ds,variant_list,
count(1) as 商品查看cnt,
count(distinct sku) as 查看SKU_cnt,
count(distinct search_query) as 查看搜索词cnt,
max(idx) as max查看位置,
max(search_times) as 搜索翻页数cnt
from all_sku_view_data_df
group by uid,ds,variant_list
"""
)


In [35]:
all_data_df = user_view_with_variant_statistics_df.merge(
    user_click_with_variant_statistics_df, on=["uid", "ds", "variant_list"], how="left"
)

In [36]:
from IPython.core.display import HTML
import pandas as pd

css = """
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@4.0.0/dist/css/bootstrap.min.css" integrity="sha384-Gn5384xqQ1aoWXA+058RXPxPg6fy4IWvTNh0E263XmFcJlSAwiGgFAW/dAiS6JXm" crossorigin="anonymous">
<style type=\"text/css\">
#abTesting table,#abTesting .table {
    color: #333;
    font-family: unset;
    font-size: 12px;
    line-height: 1.5;
    width: 90vw;
    border-collapse:
    collapse; 
    border-spacing: 0;
    font-family: "SF Pro SC", "SF Pro Text", "SF Pro Icons", "PingFang SC", "Helvetica Neue", "Helvetica", "Arial", sans-serif;
}

body{
    padding-left: 2rem;
    padding-top: 1vh;
}

tr{
    border-bottom: 1px solid #C1C3D1;
}

tr:nth-child(even) {
    background-color: #F8F8F8;
}

#abTesting td, #abTesting th {
    /* border: 1px solid transparent; No more visible border */
    height: 30px;
    padding: 0.5rem;
}

#abTesting table tbody td,#abTesting .table tbody td{
    padding: 0.1rem .75rem;
    vertical-align: middle;
}

th {
    background-color: #DFDFDF; /* Darken header a bit */
    font-weight: bolder;
    font-size: larger;
    color: #000;
    text-align: center;
}
</style>
"""


def display_p_value_below_005(row: pd.Series, p_value_col_name: str = "p_value"):
    p_value = row[p_value_col_name]
    color = "black"
    if p_value is not None and p_value <= 0.05:
        color = "red"
    return f"""<span style='font-weight:bolder;color:{color};'>{p_value}</span>"""


def display_diff_to_v1(row: pd.Series, metric: str = "diff_to_v1%"):
    diff = row[metric]
    color = "green"
    if diff is not None and float(diff) > 0.0:
        color = "red"
    return f"""<span style='font-weight:bolder;color:{color};'>{diff:.4f} %</span>"""


def dataframe_to_html(df: pd.DataFrame, title: str):
    df_to_display = df.copy()

    df_to_display["p_value"] = df_to_display.apply(display_p_value_below_005, axis=1)
    df_to_display["diff_to_v1%"] = df_to_display.apply(display_diff_to_v1, axis=1)

    html_df = df_to_display.to_html(
        escape=False, index=False, classes="table dataframe"
    )
    html_content = f"""<html><head><meta charset="UTF-8">
    <meta name="title" content="{title}">
    {css}
    </head><body>
    <h2>{title}</h2>
    <h4>当P-value <= 0.05时表示实验结果统计学显著</h4>
    <span>统计学显著时，既可能表示该试验组是好于对照组，也可能是坏于对照组</span>
    <div id="abTesting">{html_df}</div></body></html>"""

    return html_content

In [37]:
all_data_df["商品详情cnt"] = all_data_df["商品详情cnt"].fillna(0).astype(int)
all_data_df["加入购物车cnt"] = all_data_df["加入购物车cnt"].fillna(0).astype(int)
all_data_df["唤起购买cnt"] = all_data_df["唤起购买cnt"].fillna(0).astype(int)
all_data_df["avg点击位置"] = all_data_df["avg点击位置"].fillna(0.0).astype(float)
all_data_df["sku_click_rate"] = (all_data_df['商品详情cnt']*1.00/all_data_df['商品查看cnt']).fillna(0).round(5).astype(float)
all_data_df["add_cart_rate"] = (all_data_df['加入购物车cnt']/all_data_df['商品查看cnt']).fillna(0).round(5).astype(float)
all_data_df["popup_click_rate"] = (all_data_df['唤起购买cnt']/all_data_df['商品查看cnt']).fillna(0).round(5).astype(float)
print(all_data_df.columns)

Index(['uid', 'ds', 'variant_list', '商品查看cnt', '查看SKU_cnt', '查看搜索词cnt', 'max查看位置', '搜索翻页数cnt', '商品详情cnt', '加入购物车cnt', '唤起购买cnt', 'avg点击位置', 'max点击位置', 'min点击位置', '点击SKU_cnt', '搜索词cnt', 'sku_click_rate', 'add_cart_rate', 'popup_click_rate'], dtype='object')


In [38]:
import pandas as pd
from scipy.stats import ttest_ind


def calculate_p_values(
    df: pd.DataFrame,
    metric: str = "商品详情cnt",
) -> pd.DataFrame:
    """
    Calculate p-values for each combination of category1 and page_name.
    Compares metric between control group (V1) and each of V2, V3, V4.

    Parameters:
    - df (pd.DataFrame): The input DataFrame containing A/B test data.
    - metric (str): The metric column to be analyzed (default is 'added_quantity').

    Returns:
    - pd.DataFrame: A DataFrame with category1, page_name, variant, p-value, and statistics columns.
    """
    p_values = []

    control = df[df["variant_list"] == "V2"][metric]
    control_avg = control.mean()

    for variant in ["V1", "V2", "V3", "V4", "V5"]:  
        # ["V1", "V2", "V3", "V4", "V5"]: 
        # df["variant_list"].unique()
        # Separate each test variant (V1, V2, V3, V4)
        test_group = df[df["variant_list"] == variant]
        test = test_group[metric]

        print(f'variant:{variant}, test_group ds length: {len(test_group["ds"].unique())}')
        if len(test_group["ds"].unique())<=0:
            continue

        # Calculate statistics
        stats = {
            "均值": round(test.mean(), 4),
            "std": round(test.std(), 4),
            "diff_to_v1%": round(100.00*(test.mean() - control_avg) / control_avg,2),
            "q50": test.quantile(0.5),
            "q75": test.quantile(0.75),
            "q90": test.quantile(0.9),
            "q95": test.quantile(0.95),
            "q97": test.quantile(0.97),
            "q99": test.quantile(0.99),
            "q995": test.quantile(0.995),
            "max": test.max(),
            "日均总数": round(test.sum() / len(test_group["ds"].unique())),
            "日均实验UV": round(
                len(test_group[["uid", "ds"]].drop_duplicates())
                / len(test_group["ds"].unique())
            ),
            "日均转化UV": round(
                len(
                    test_group[test_group[metric] > 0][
                        ["uid", "ds"]
                    ].drop_duplicates()
                )
                / len(test_group["ds"].unique())
            ),
            "日期范围": f"{test_group['ds'].min()}~{test_group['ds'].max()}".replace(
                "2025", ""
            ),
            "metric": metric,
        }

        # Ensure both groups have enough data for a valid t-test
        if len(control) > 1 and len(test) > 1:
            # Perform independent t-test
            stat, p_val = ttest_ind(control, test, equal_var=False)
            p_values.append(
                {
                    "variant_list": variant,
                    "p_value": round(p_val, 4),
                    **stats,
                }
            )
        else:
            # Not enough data for statistical testing
            p_values.append(
                {
                    "variant_list": variant,
                    "p_value": None,
                    **stats,
                }
            )

    return pd.DataFrame(p_values)

In [39]:
# Define the desired order for sorting
variant_order = ["V1", "V2", "V3", "V4", "V5"]


# Create a custom sort key function
def sort_key(variant):
    # Split variant by commas
    parts = variant.split(",")
    # Determine the order based on the first variant in the list
    if parts[0] in variant_order:
        return variant_order.index(parts[0])
    else:
        return len(variant_order)  # Place all other variants after V1, V2, V3, V4


metrics_list = [
    "sku_click_rate",
    "add_cart_rate",
    "popup_click_rate",
    "商品详情cnt",
    "加入购物车cnt",
    "唤起购买cnt",
    "搜索翻页数cnt",
    "avg点击位置",
    "min点击位置",
]

all_p_values_df = pd.DataFrame()
for metric in metrics_list:
    p_values_df = calculate_p_values(
        all_data_df,
        metric=metric,
    )

    p_values_df["variant_list"] = p_values_df["variant_list"].apply(
        lambda x: x if x in variant_order else "X_" + x
    )
    p_values_df = p_values_df.sort_values(
        by="variant_list", key=lambda x: x.map(sort_key)
    )
    p_values_df["variant_list"] = p_values_df["variant_list"].str.replace("X_", "")

    title = f"搜索AB--{metric}_p-value分布-{p_values_df.iloc[0]['日期范围']}"

    html_content = dataframe_to_html(df=p_values_df, title=title)
    file_path = f"./data/{title}.html"

    # 保存HTML到本地文件：
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(html_content)

    print(f"写入HTML成功！{file_path}")
    all_p_values_df = pd.concat([all_p_values_df, p_values_df], ignore_index=True)
    title_all = f"搜索AB--指标全集_p-value分布-{p_values_df.iloc[0]['日期范围']}"
    html_content = dataframe_to_html(df=all_p_values_df, title=title_all);
    file_path = f"./data/{title_all}.html"

    # 保存HTML到本地文件：
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(html_content)

    print(f"写入HTML成功！{file_path}")

all_p_values_df

variant:V1, test_group ds length: 5
variant:V2, test_group ds length: 5
variant:V3, test_group ds length: 5
variant:V4, test_group ds length: 5
variant:V5, test_group ds length: 5
写入HTML成功！./data/搜索AB--sku_click_rate_p-value分布-0101~0105.html
写入HTML成功！./data/搜索AB--指标全集_p-value分布-0101~0105.html
variant:V1, test_group ds length: 5
variant:V2, test_group ds length: 5
variant:V3, test_group ds length: 5
variant:V4, test_group ds length: 5
variant:V5, test_group ds length: 5
写入HTML成功！./data/搜索AB--add_cart_rate_p-value分布-0101~0105.html
写入HTML成功！./data/搜索AB--指标全集_p-value分布-0101~0105.html
variant:V1, test_group ds length: 5
variant:V2, test_group ds length: 5
variant:V3, test_group ds length: 5
variant:V4, test_group ds length: 5
variant:V5, test_group ds length: 5
写入HTML成功！./data/搜索AB--popup_click_rate_p-value分布-0101~0105.html
写入HTML成功！./data/搜索AB--指标全集_p-value分布-0101~0105.html
variant:V1, test_group ds length: 5
variant:V2, test_group ds length: 5
variant:V3, test_group ds length: 5
variant:V

,variant_list,p_value,均值,std,diff_to_v1%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.1731,0.0410,0.0723,4.63,0.000000,0.055560,0.125000,0.200000,0.250,0.250000,0.333330,1.00000,46,1130,532,0101~0105,sku_click_rate
1,V2,1.0000,0.0391,0.0676,0.00,0.000000,0.052630,0.125000,0.200000,0.250,0.250000,0.283568,1.50000,43,1087,506,0101~0105,sku_click_rate
2,V3,0.0070,0.0428,0.0733,9.25,0.000000,0.058820,0.128292,0.222220,0.250,0.250000,0.333330,1.00000,48,1116,531,0101~0105,sku_click_rate
3,V4,0.2971,0.0405,0.0691,3.46,0.000000,0.055560,0.125000,0.200000,0.250,0.250000,0.326664,1.00000,46,1133,533,0101~0105,sku_click_rate
4,V5,0.1004,0.0413,0.0685,5.44,0.000000,0.055560,0.125000,0.200000,0.250,0.250000,0.333330,0.60000,46,1123,531,0101~0105,sku_click_rate
5,V1,0.9503,0.0327,0.0686,-0.24,0.000000,0.038460,0.111110,0.175318,0.250,0.250000,0.333330,1.50000,37,1130,427,0101~0105,add_cart_rate
6,V2,1.0000,0.0328,0.0638,0.00,0.000000,0.041670,0.111110,0.166670,0.250,0.250000,0.285710,0.75000,36,1087,409,0101~0105,add_cart_rate
7,V3,0.6942,0.0323,0.0656,-1.48,0.000000,0.035710,0.111110,0.200000,0.250,0.250000,0.333330,0.75000,36,1116,405,0101~0105,add_cart_rate
8,V4,0.5886,0.0321,0.0792,-2.25,0.000000,0.036360,0.108110,0.166670,0.250,0.250000,0.270791,2.94118,36,1133,419,0101~0105,add_cart_rate
9,V5,0.3053,0.0316,0.0636,-3.79,0.000000,0.035710,0.111110,0.182663,0.250,0.250000,0.330830,0.75000,35,1123,412,0101~0105,add_cart_rate


In [40]:
from odps_client import get_odps_sql_result_as_df, write_pandas_df_into_odps

partition_spec = f"pt={datetime.now().strftime('%Y%m%d')}"

write_pandas_df_into_odps(
    df=all_user_variant_df,
    table_name="summerfarm_ds.temp_search_ab_user_variant_df",
    partition_spec=partition_spec,
    overwrite=True,
    lifecycle=30,
)

order_query = """
with user_orders as (
    SELECT  m_id
        ,total_price
        ,order_no
        ,DATE_FORMAT(order_time,'yyyyMMdd') as order_date
    FROM    summerfarm_tech.ods_orders_df
    WHERE   ds = MAX_PT('summerfarm_tech.ods_orders_df')
    AND     status IN (2,3,6)
    AND     order_time >= '2025-01-01 00:00:00'
    AND     m_size = '单店'
),user_variants as (
    select ds as event_date,uid,variant_list
    from summerfarm_ds.temp_search_ab_user_variant_df
    where pt=max_pt('summerfarm_ds.temp_search_ab_user_variant_df')
)
select a.event_date,a.uid,a.variant_list,sum(b.total_price) as order_gmv,
    count(b.order_no) as order_cnt,round(sum(b.total_price)/count(b.order_no),2) as avg_order_gmv
from user_variants a
left join user_orders b on a.uid=b.m_id and a.event_date=b.order_date
group by a.event_date,a.uid,a.variant_list
"""

user_orders_df = get_odps_sql_result_as_df(order_query)
user_orders_df.head(2)

2025-01-05 17:10:08 - INFO - DaraFrame字段合集:type,page_ame,search_times,experiment_id,pt,create_time,uid,api,variant_list,ds
2025-01-05 17:10:12 - INFO - Tunnel session created: <TableUploadSession id=20250105171012c4d9c20b11364047 project=summerfarm_ds table=temp_search_ab_user_variant_df partition_spec=pt=20250105>
2025-01-05 17:10:15 - INFO - 成功写入odps:summerfarm_ds.temp_search_ab_user_variant_df, partition_spec:pt=20250105, attemp:0
2025-01-05 17:10:23 - INFO - Tunnel session created: <InstanceDownloadSession id=20250105171023c415696411230c08 project_name=summerfarm_ds instance_id=20250105091015735g8mdopeyma>
2025-01-05 17:10:24 - INFO - sql:

with user_orders as (
    SELECT  m_id
        ,total_price
        ,order_no
        ,DATE_FORMAT(order_time,'yyyyMMdd') as order_date
    FROM    summerfarm_tech.ods_orders_df
    WHERE   ds = MAX_PT('summerfarm_tech.ods_orders_df')
    AND     status IN (2,3,6)
    AND     order_time >= '2025-01-01 00:00:00'
    AND     m_size = '单店'
),user_v

,event_date,uid,variant_list,order_gmv,order_cnt,avg_order_gmv
0,20250102,465338,V2,138,1,138
1,20250102,441785,V1,336.4,1,336.4


In [41]:
user_orders_df["order_gmv"] = user_orders_df["order_gmv"].fillna(0)
user_orders_df["order_gmv"] = user_orders_df["order_gmv"].astype(float)

user_orders_df["avg_order_gmv"] = user_orders_df["avg_order_gmv"].fillna(0.0)
user_orders_df["avg_order_gmv"] = user_orders_df["avg_order_gmv"].astype(float)

user_orders_df["category1"] = "ignore"
user_orders_df["page_name"] = "ignore"

user_orders_during_ab_df = user_orders_df[
    user_orders_df["variant_list"].isin(["V1", "V2", "V3", "V4", "V5"])
]
user_orders_during_ab_df.rename(columns={"event_date": "ds"}, inplace=True)

all_order_pvalue_df = pd.DataFrame()
for metric in ["order_gmv", "avg_order_gmv", "order_cnt"]:
    gmv_df = calculate_p_values(df=user_orders_during_ab_df, metric=metric)
    display(gmv_df)
    all_order_pvalue_df = pd.concat([all_order_pvalue_df, gmv_df], ignore_index=True)

title = f"搜索AB--订单转化p-value分布-{all_order_pvalue_df.iloc[0]['日期范围']}"
html_content = dataframe_to_html(df=all_order_pvalue_df, title=title)
file_path = f"./data/{title}.html"

# 保存HTML到本地文件：
with open(file_path, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"写入HTML成功！{file_path}")
display(all_order_pvalue_df)

variant:V1, test_group ds length: 5
variant:V2, test_group ds length: 5
variant:V3, test_group ds length: 5
variant:V4, test_group ds length: 5
variant:V5, test_group ds length: 5


/var/folders/b3/9hcz86fx1_z_8m4121xwbs2h0000gn/T/ipykernel_92676/2609040990.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  user_orders_during_ab_df.rename(columns={"event_date": "ds"}, inplace=True)


,variant_list,p_value,均值,std,diff_to_v1%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.0675,289.4436,803.7776,-11.87,0.0,298.75,747.50,1168.75,1582.125,3050.450,4531.43275,20168.0,329155,1137,529,0101~0105,order_gmv
1,V2,1.0000,328.4379,1364.5270,0.00,0.0,305.00,732.40,1177.10,1634.820,3122.860,5825.85000,59300.0,358589,1092,513,0101~0105,order_gmv
2,V3,0.2492,301.7809,1044.1489,-8.12,0.0,283.40,752.00,1247.25,1710.100,3029.300,5096.40000,41095.0,338960,1123,511,0101~0105,order_gmv
3,V4,0.0246,280.6419,794.5953,-14.55,0.0,292.00,745.64,1182.00,1640.000,2828.240,4337.28000,25620.0,319763,1139,514,0101~0105,order_gmv
4,V5,0.0279,280.7683,853.0909,-14.51,0.0,289.00,726.20,1150.00,1597.240,2822.064,3977.27760,33600.0,317212,1130,512,0101~0105,order_gmv


variant:V1, test_group ds length: 5
variant:V2, test_group ds length: 5
variant:V3, test_group ds length: 5
variant:V4, test_group ds length: 5
variant:V5, test_group ds length: 5


,variant_list,p_value,均值,std,diff_to_v1%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.1439,234.4364,622.9184,-8.27,0.0,258.000,621.0,977.7500,1239.1950,2241.45,3023.31250,16800.0,266601,1137,529,0101~0105,avg_order_gmv
1,V2,1.0000,255.5709,876.9408,0.00,0.0,263.500,615.0,952.4000,1239.7558,2422.05,4192.75000,29650.0,279032,1092,513,0101~0105,avg_order_gmv
2,V3,0.3590,241.8357,683.9874,-5.37,0.0,254.625,628.0,1007.6675,1376.6500,2429.95,3447.63425,21000.0,271630,1123,511,0101~0105,avg_order_gmv
3,V4,0.1844,235.3740,718.6321,-7.90,0.0,260.000,602.8,952.0000,1220.0000,2259.16,3739.44000,25620.0,268185,1139,514,0101~0105,avg_order_gmv
4,V5,0.0114,220.8863,513.8262,-13.57,0.0,254.000,610.0,942.0000,1187.8000,2218.56,2828.28000,11200.0,249557,1130,512,0101~0105,avg_order_gmv


variant:V1, test_group ds length: 5
variant:V2, test_group ds length: 5
variant:V3, test_group ds length: 5
variant:V4, test_group ds length: 5
variant:V5, test_group ds length: 5


,variant_list,p_value,均值,std,diff_to_v1%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.6143,0.5760,0.7480,-1.22,0.0,1.0,1.0,2.0,2.0,3.0,3.0,9,655,1137,529,0101~0105,order_cnt
1,V2,1.0000,0.5831,0.7388,0.00,0.0,1.0,1.0,2.0,2.0,3.0,3.0,7,637,1092,513,0101~0105,order_cnt
2,V3,0.0749,0.5582,0.7291,-4.26,0.0,1.0,1.0,2.0,2.0,3.0,3.0,8,627,1123,511,0101~0105,order_cnt
3,V4,0.0822,0.5587,0.7409,-4.18,0.0,1.0,1.0,2.0,2.0,3.0,4.0,7,637,1139,514,0101~0105,order_cnt
4,V5,0.1941,0.5649,0.7378,-3.12,0.0,1.0,1.0,2.0,2.0,3.0,4.0,6,638,1130,512,0101~0105,order_cnt


写入HTML成功！./data/搜索AB--订单转化p-value分布-0101~0105.html


,variant_list,p_value,均值,std,diff_to_v1%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.0675,289.4436,803.7776,-11.87,0.0,298.750,747.50,1168.7500,1582.1250,3050.450,4531.43275,20168.0,329155,1137,529,0101~0105,order_gmv
1,V2,1.0000,328.4379,1364.5270,0.00,0.0,305.000,732.40,1177.1000,1634.8200,3122.860,5825.85000,59300.0,358589,1092,513,0101~0105,order_gmv
2,V3,0.2492,301.7809,1044.1489,-8.12,0.0,283.400,752.00,1247.2500,1710.1000,3029.300,5096.40000,41095.0,338960,1123,511,0101~0105,order_gmv
3,V4,0.0246,280.6419,794.5953,-14.55,0.0,292.000,745.64,1182.0000,1640.0000,2828.240,4337.28000,25620.0,319763,1139,514,0101~0105,order_gmv
4,V5,0.0279,280.7683,853.0909,-14.51,0.0,289.000,726.20,1150.0000,1597.2400,2822.064,3977.27760,33600.0,317212,1130,512,0101~0105,order_gmv
5,V1,0.1439,234.4364,622.9184,-8.27,0.0,258.000,621.00,977.7500,1239.1950,2241.450,3023.31250,16800.0,266601,1137,529,0101~0105,avg_order_gmv
6,V2,1.0000,255.5709,876.9408,0.00,0.0,263.500,615.00,952.4000,1239.7558,2422.050,4192.75000,29650.0,279032,1092,513,0101~0105,avg_order_gmv
7,V3,0.3590,241.8357,683.9874,-5.37,0.0,254.625,628.00,1007.6675,1376.6500,2429.950,3447.63425,21000.0,271630,1123,511,0101~0105,avg_order_gmv
8,V4,0.1844,235.3740,718.6321,-7.90,0.0,260.000,602.80,952.0000,1220.0000,2259.160,3739.44000,25620.0,268185,1139,514,0101~0105,avg_order_gmv
9,V5,0.0114,220.8863,513.8262,-13.57,0.0,254.000,610.00,942.0000,1187.8000,2218.560,2828.28000,11200.0,249557,1130,512,0101~0105,avg_order_gmv
